In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split,KFold
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#/kaggle/input/q1-ka-ai-2026/Q1_data.csv
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:

df.head(10)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)
  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)
df

In [ ]:
# Task 2: Write your code here:
#calculting the missing percentage so that i thing what i do with them
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)
print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
df_clean = df.copy() #first lets copy
df_clean = df_clean.dropna(subset=["Delivery_Time"]) #drop the na smaples in target

columns_to_mode = ['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']

for col in columns_to_mode:
  df_clean[col]=df_clean[col].fillna(df_clean[col].mode()[0]) #i  want descret numbers





In [ ]:
df_clean.head()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_clean):
  # duplicated().sum(): Counts identical rows.
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")

  if duplicates > 0:
    print("Dropping Duplicates...")
    # inplace=True: Modifies the actual DataFrame, doesn't return a copy.
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df_clean.shape

In [ ]:
# Task 4: Write your code here:
catogerical = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
le = LabelEncoder()
for col in catogerical:
  df_clean[col]=le.fit_transform(df_clean[col])
df_clean.head()

In [ ]:
#i want to do scale on the train only
X = df_clean.drop('Delivery_Time',axis=1)
y= df_clean['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Task 5: Write your code here:
standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
# even if its imbalance, we dont care it is regression
target=df_clean['Delivery_Time'].value_counts()
target

In [ ]:
# Task 1: Write your code here:
# i did it above becaus i want to do the scalte on the train only so no data leake
#i used kfold only becuse it is regression

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

model.fit(X_train_scaled, y_train)
print("Model trained!")

y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  ${mae:,.2f}")



kfold = KFold(n_splits=3, shuffle=True, random_state=42)
mae_scores = []
for train_idx, val_idx in kfold.split(X_train_scaled):
    # Slice the arrays
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics for this specific fold
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
mae_scores = np.array(mae_scores)
print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:

feature_cols=['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('predicted Distribution')
plt.xlabel('time in mins')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: